# 스파이크 후속: 튜닝 후 XGBoost가 로지스틱 회귀 대비 효용이 있는가

배경: `spike_feasibility.ipynb`에서 튜닝하지 않은 기본값 XGBoost(AUC 0.8490)가
로지스틱 회귀 베이스라인(AUC 0.8597)보다 낮게 나왔다. "튜닝하고 로지스틱과 비교해 최종 결정하자"는
결정에 따라, 두 모델 모두 하이퍼파라미터 탐색을 거친 뒤 공정하게 비교한다.

범위: 빠른 판단을 위한 스파이크다. 정식 Phase 3(교차검증 설계, KS, 구간별 연체율, 모델 아티팩트 저장 등)를
대체하지 않는다. 전처리도 이전 스파이크와 동일한 가안(중앙값 대체 + winsorize)을 그대로 쓴다.


In [1]:
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
DATA_DIR = "../../data/raw"
TARGET = "SeriousDlqin2yrs"
DELINQ_COLS = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

timings = {}


@contextmanager
def timer(step_name):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    timings[step_name] = elapsed
    print(f"[{step_name}] {elapsed:.1f}초")


## 1. 데이터 로드 + 최소 전처리 (이전 스파이크와 동일)

In [2]:
with timer("로드+전처리"):
    train = pd.read_csv(f"{DATA_DIR}/cs-training.csv", index_col=0)
    df = train.copy()

    for col in DELINQ_COLS:
        df.loc[df[col] >= 96, col] = np.nan

    missing_cols = ["MonthlyIncome", "NumberOfDependents"] + DELINQ_COLS
    df[missing_cols] = df[missing_cols].fillna(df[missing_cols].median())

    feature_cols = [c for c in df.columns if c != TARGET]
    lower = df[feature_cols].quantile(0.005)
    upper = df[feature_cols].quantile(0.995)
    df[feature_cols] = df[feature_cols].clip(lower=lower, upper=upper, axis=1)

X = df[feature_cols]
y = df[TARGET]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("학습:", X_train.shape, "홀드아웃:", X_holdout.shape)


[로드+전처리] 0.1초
학습: (120000, 10) 홀드아웃: (30000, 10)


## 2. 로지스틱 회귀 — 하이퍼파라미터 탐색

`C`(정규화 강도)와 `class_weight` 조합을 5-fold CV로 탐색한다.


In [3]:
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

logreg_grid = {
    "clf__C": [0.001, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
    "clf__class_weight": [None, "balanced"],
}

with timer("로지스틱 회귀 탐색"):
    logreg_search = GridSearchCV(
        logreg_pipe, logreg_grid, scoring="roc_auc", cv=cv, n_jobs=-1
    )
    logreg_search.fit(X_train, y_train)

print("최적 파라미터:", logreg_search.best_params_)
print("CV AUC:", round(logreg_search.best_score_, 4))

logreg_best = logreg_search.best_estimator_
logreg_proba = logreg_best.predict_proba(X_holdout)[:, 1]
logreg_auc = roc_auc_score(y_holdout, logreg_proba)
print("홀드아웃 AUC:", round(logreg_auc, 4))


[로지스틱 회귀 탐색] 2.2초
최적 파라미터: {'clf__C': 0.001, 'clf__class_weight': 'balanced'}
CV AUC: 0.8542
홀드아웃 AUC: 0.8598


## 3. XGBoost — 하이퍼파라미터 탐색

`scale_pos_weight`를 1(미적용)부터 실제 불균형 비율까지 포함해 탐색한다
(이전 스파이크에서 `scale_pos_weight` 적용이 랭킹을 오히려 낮췄을 가능성이 있었기 때문).


In [4]:
pos_weight_ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "scale_pos_weight": [1, 5, 10, round(pos_weight_ratio, 2), 20],
}

with timer("XGBoost 탐색"):
    xgb_search = RandomizedSearchCV(
        xgb.XGBClassifier(
            objective="binary:logistic", eval_metric="auc", random_state=RANDOM_STATE
        ),
        param_distributions=xgb_param_dist,
        n_iter=40,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_search.fit(X_train, y_train)

print("최적 파라미터:", xgb_search.best_params_)
print("CV AUC:", round(xgb_search.best_score_, 4))

xgb_best = xgb_search.best_estimator_
xgb_proba = xgb_best.predict_proba(X_holdout)[:, 1]
xgb_auc = roc_auc_score(y_holdout, xgb_proba)
print("홀드아웃 AUC:", round(xgb_auc, 4))


[XGBoost 탐색] 18.4초
최적 파라미터: {'colsample_bytree': np.float64(0.6888431241882921), 'learning_rate': np.float64(0.044760956526768016), 'max_depth': 5, 'min_child_weight': 8, 'n_estimators': 259, 'scale_pos_weight': 1, 'subsample': np.float64(0.7615344684232164)}
CV AUC: 0.8647
홀드아웃 AUC: 0.8691


## 4. 비교

In [5]:
result = pd.DataFrame(
    [
        ("로지스틱 회귀 (튜닝)", logreg_search.best_score_, logreg_auc),
        ("XGBoost (튜닝)", xgb_search.best_score_, xgb_auc),
    ],
    columns=["모델", "CV AUC (5-fold, 학습셋)", "홀드아웃 AUC"],
)
result["홀드아웃 AUC 차이(XGB-LR)"] = result["홀드아웃 AUC"] - logreg_auc
result


,모델,"CV AUC (5-fold, 학습셋)",홀드아웃 AUC,홀드아웃 AUC 차이(XGB-LR)
0,로지스틱 회귀 (튜닝),0.854233,0.859758,0.000000
1,XGBoost (튜닝),0.864683,0.869144,0.009386


In [6]:
timing_df = pd.DataFrame(
    [(k, f"{v:.1f}초") for k, v in timings.items()], columns=["단계", "소요 시간"]
)
print(f"전체 합계: {sum(timings.values()):.1f}초")
timing_df


전체 합계: 20.7초


,단계,소요 시간
0,로드+전처리,0.1초
1,로지스틱 회귀 탐색,2.2초
2,XGBoost 탐색,18.4초


## 5. AUC 기준 판단

위 표의 "홀드아웃 AUC 차이" 값으로 판단한다.

- 차이가 실질적으로 크면(관례적으로 +0.01~0.02 이상) → XGBoost 우위가 뚜렷하다고 본다.
- 차이가 미미하면 → AUC만으로는 우열을 가리기 애매하다. 상위 위험군 포착 성능(6절)도 함께 봐야 한다.


## 6. 고위험 상위 5% 포착 성능

실무적으로 중요한 질문: 두 모델이 매긴 점수로 **예측 부실 확률이 가장 높은 상위 5%**를 골라냈을 때,
- 그 안에 실제 부실자가 얼마나 섞여 있는지 (정밀도 — "상위 5%로 찍었는데 실제로 맞았나")
- 전체 부실자 중 몇 %를 그 상위 5% 안에서 잡아내는지 (포착률/재현율 — "실제 부실자를 놓치지 않고 상위권에 몰아넣었나")
- 무작위로 5%를 뽑았을 때 대비 몇 배 더 부실자가 몰려있는지 (lift)

를 홀드아웃 기준으로 비교한다.


In [7]:
def top_k_metrics(y_true, proba, k_ratio=0.05):
    n = len(y_true)
    k = int(np.ceil(n * k_ratio))
    order = np.argsort(-proba)  # 위험도 내림차순
    top_idx = order[:k]

    y_true_arr = np.asarray(y_true)
    n_bad_total = y_true_arr.sum()
    n_bad_in_top = y_true_arr[top_idx].sum()

    precision = n_bad_in_top / k          # 상위 5% 중 실제 부실 비율
    recall = n_bad_in_top / n_bad_total   # 전체 부실자 중 상위 5%가 잡아낸 비율
    base_rate = y_true_arr.mean()
    lift = precision / base_rate

    return {
        "표본 수(상위 5%)": k,
        "상위 5% 내 실제 부실자 수": int(n_bad_in_top),
        "정밀도(상위 5% 중 실제 부실 비율)": precision,
        "포착률(전체 부실자 중 상위 5%가 잡아낸 비율)": recall,
        "lift(전체 평균 대비 배수)": lift,
    }


top5_logreg = top_k_metrics(y_holdout, logreg_proba)
top5_xgb = top_k_metrics(y_holdout, xgb_proba)

top5_table = pd.DataFrame([top5_logreg, top5_xgb], index=["로지스틱 회귀 (튜닝)", "XGBoost (튜닝)"]).T
top5_table


,로지스틱 회귀 (튜닝),XGBoost (튜닝)
표본 수(상위 5%),1500.000000,1500.000000
상위 5% 내 실제 부실자 수,712.000000,720.000000
정밀도(상위 5% 중 실제 부실 비율),0.474667,0.480000
포착률(전체 부실자 중 상위 5%가 잡아낸 비율),0.355112,0.359102
lift(전체 평균 대비 배수),7.102244,7.182045


In [8]:
print(f"전체 홀드아웃 부실률(기준선): {y_holdout.mean():.2%}")
print(f"로지스틱 회귀 상위 5% 부실률: {top5_logreg['정밀도(상위 5% 중 실제 부실 비율)']:.2%} "
      f"(lift {top5_logreg['lift(전체 평균 대비 배수)']:.2f}배)")
print(f"XGBoost   상위 5% 부실률: {top5_xgb['정밀도(상위 5% 중 실제 부실 비율)']:.2%} "
      f"(lift {top5_xgb['lift(전체 평균 대비 배수)']:.2f}배)")
print()
print(f"로지스틱 회귀 상위 5% 포착률: {top5_logreg['포착률(전체 부실자 중 상위 5%가 잡아낸 비율)']:.2%}")
print(f"XGBoost   상위 5% 포착률: {top5_xgb['포착률(전체 부실자 중 상위 5%가 잡아낸 비율)']:.2%}")


전체 홀드아웃 부실률(기준선): 6.68%
로지스틱 회귀 상위 5% 부실률: 47.47% (lift 7.10배)
XGBoost   상위 5% 부실률: 48.00% (lift 7.18배)

로지스틱 회귀 상위 5% 포착률: 35.51%
XGBoost   상위 5% 포착률: 35.91%


## 7. 종합 결론

AUC(5절)와 상위 5% 포착 성능(6절)을 함께 본다.

- 두 지표 모두 XGBoost가 앞서면 → XGBoost 최종 채택 근거로 충분.
- AUC 차이는 작은데 상위 5% 포착 성능 차이가 크면(또는 반대라면) → 실제 운영 목적(전체 순위 판별 vs 최상위 위험군 선별)에
  맞춰 지표를 다시 정하고 판단해야 한다.
- 최종 모델 결정은 사용자 몫이며, `CLAUDE.md`·`docs/project-plan.md`에 반영한다.
